In [51]:
!pip install pandas -q
import math
import pandas as pd

# ======================
# 1. Hàm tính khoảng cách
# ======================
def distance(lat1, lon1, lat2, lon2):
    return math.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2)

# ======================
# 2. Dữ liệu kho
# ======================
depots = pd.DataFrame([
    {"id": "D1", "lat": 10.7626, "lon": 106.6601},
    {"id": "D2", "lat": 10.7680, "lon": 106.6680},
])

# ======================
# 3. Dữ liệu điểm giao
# ======================
deliveries = pd.DataFrame([
    {"id": 1, "lat": 10.7612, "lon": 106.6571},
    {"id": 2, "lat": 10.7629, "lon": 106.6630},
    {"id": 3, "lat": 10.7655, "lon": 106.6601},
    {"id": 4, "lat": 10.7595, "lon": 106.6549},
    {"id": 5, "lat": 10.7660, "lon": 106.6660},
    {"id": 6, "lat": 10.7582, "lon": 106.6615},
])

print("Kho hàng:")
print(depots)

print("\nĐiểm giao:")
print(deliveries)

# ======================
# 4. Gán mỗi điểm giao cho kho gần nhất
# ======================
assigned = []

for _, d in deliveries.iterrows():
    best_depot = None
    best_dist = 999

    for _, depot in depots.iterrows():
        dist = distance(d["lat"], d["lon"], depot["lat"], depot["lon"])
        if dist < best_dist:
            best_dist = dist
            best_depot = depot["id"]

    assigned.append(best_depot)

deliveries["assigned_depot"] = assigned

print("\nSau khi gán kho gần nhất:")
print(deliveries)

# ======================
# 5. Tạo tuyến giao đơn giản
# đi theo điểm gần nhất tiếp theo
# ======================
routes = {}

for depot_id in depots["id"]:
    depot = depots[depots["id"] == depot_id].iloc[0]
    group = deliveries[deliveries["assigned_depot"] == depot_id].copy()

    unvisited = group.to_dict("records")
    current_lat = depot["lat"]
    current_lon = depot["lon"]

    route = [depot_id]
    total_distance = 0

    while len(unvisited) > 0:
        nearest = None
        nearest_dist = 999

        for p in unvisited:
            dist = distance(current_lat, current_lon, p["lat"], p["lon"])
            if dist < nearest_dist:
                nearest_dist = dist
                nearest = p

        route.append(nearest["id"])
        total_distance += nearest_dist
        current_lat = nearest["lat"]
        current_lon = nearest["lon"]
        unvisited.remove(nearest)

    # quay về kho
    back_dist = distance(current_lat, current_lon, depot["lat"], depot["lon"])
    total_distance += back_dist
    route.append(depot_id)

    routes[depot_id] = {
        "route": route,
        "distance": total_distance
    }

# ======================
# 6. In kết quả
# ======================
print("\nKết quả tuyến giao:")
for k, v in routes.items():
    print(k, "->", v["route"], "| Tổng quãng đường:", round(v["distance"], 4))


Kho hàng:
   id      lat       lon
0  D1  10.7626  106.6601
1  D2  10.7680  106.6680

Điểm giao:
   id      lat       lon
0   1  10.7612  106.6571
1   2  10.7629  106.6630
2   3  10.7655  106.6601
3   4  10.7595  106.6549
4   5  10.7660  106.6660
5   6  10.7582  106.6615

Sau khi gán kho gần nhất:
   id      lat       lon assigned_depot
0   1  10.7612  106.6571             D1
1   2  10.7629  106.6630             D1
2   3  10.7655  106.6601             D1
3   4  10.7595  106.6549             D1
4   5  10.7660  106.6660             D2
5   6  10.7582  106.6615             D1

Kết quả tuyến giao:
D1 -> ['D1', 3, 2, 6, 1, 4, 'D1'] | Tổng quãng đường: 0.0259
D2 -> ['D2', 5, 'D2'] | Tổng quãng đường: 0.0057
